In [4]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from time import sleep
import os
import requests
from bs4 import BeautifulSoup
import re
import requests
from bs4 import BeautifulSoup
import urllib3

# Disable SSL warnings and skip certificate verification because the server certificate
# cannot be verified in this environment.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [5]:

# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'CY ICCS'
print(f"Running {regulatorName} Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename= f'{regulatorName} SQL Ready {str(now).replace(":",".")[:-7]}.xlsx'

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)





Running CY ICCS Web Scraping Tool v.1.1


In [6]:

# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------


def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict




In [7]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = {
    regulatorName+' 1' : 'https://www.mof.gov.cy/mof/iccs.nsf/table1_en/table1_en?openform',

    }

Typology = {

            regulatorName+" 1": "Register of Insurance/Reinsurance Undertakings",

            }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

          'Phone - Mother company': [], 'Check': []}





processdate = now.strftime('%Y-%m-%d')



In [8]:

# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

#driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=chromeOptions )

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



In [9]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for index, reg in enumerate(regdict):

    print(f"[INFO] : Working {index+1}/{len(regdict)} _({reg})_ ")

    sleep(3)
    # Fetch the page content
    response.encoding = 'utf-8'  # Ensure correct encoding
    response = requests.get(regdict[reg], verify=False)
    soup = BeautifulSoup(response.content, 'html.parser')

    tables = soup.find_all('table')

    for table_index, table in enumerate(tables):

        headers = [th.get_text(strip=True) for th in table.find_all('th')]
        rows = table.find_all('tr')[1:]  # Skip header row

        for row_index, row in enumerate(rows):
            cells = row.find_all(['td', 'th'])
            values = [cell.get_text(strip=True) for cell in cells]

            print(f"\nRow {row_index + 1}:")
            for col_index, header in enumerate(headers):
                var_name = header.replace(" ", "_").lower()
                var_value = values[col_index] if col_index < len(values) else ''
                print(f"{var_name} = '{var_value}'")
                if var_name == 'licence_number':
                    sqldict['InternalID_1_type'].append('Licence Number')
                    sqldict['InternalID_1'].append(var_value)
                elif var_name == 'name_of_undertaking':
                    sqldict['Name'].append(var_value)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
                elif var_name == 'type_of_undertaking':
                    sqldict['Typology'].append(var_value)
                elif var_name == 'lei':
                    sqldict['LEI Code'].append(var_value)
                elif var_name == 'country_of_authorisation':
                    sqldict['Cntry'].append(var_value)

            sqldict = bourange_same_length_array(sqldict)


[INFO] : Working 1/1 _(CY ICCS 1)_ 

Row 1:
name_of_undertaking = 'INTERNATIONAL TRANSPORT INTERMEDIARIES INSURANCE COMPANY (EUROPE) LIMITED'
country_of_authorisation = 'Cyprus'
licence_number = '184'
lei = '2138004D2YA2PY9CLN46'
type_of_undertaking = 'Undertaking pursuing non-life insurance activity'
classes_of_business_-_life = '-'
classes_of_business_-_non_life = '13'

Row 2:
name_of_undertaking = 'AMERICAN STEAMSHIP OWNERS MARINE INSURANCE COMPANY (EUROPE) LIMITED (FORMER AMERICAN HELLENIC HULL INSURANCE COMPANY LIMITED)'
country_of_authorisation = 'Cyprus'
licence_number = '180'
lei = '213800C4MZ2GF4ZINM93'
type_of_undertaking = 'Undertaking pursuing non-life insurance activity'
classes_of_business_-_life = '-'
classes_of_business_-_non_life = '1,6,7,12,17'

Row 3:
name_of_undertaking = 'ANCORIA INSURANCE PUBLIC LTD'
country_of_authorisation = 'Cyprus'
licence_number = '97'
lei = '2138004X3TKZETJ8O713'
type_of_undertaking = 'Undertaking pursuing life insurance activity'
classes_of

In [ ]:

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)
df.to_excel(filename,index=False)
driver.quit()